# Modulo 1 — AI Agents per Big Data & ETL Operations

**Caso pratico:** Costruzione di un *Autonomous Big Data Pipeline Operations Agent* capace di diagnosticare un incidente in una pipeline ETL di produzione (PySpark / SQL / Delta Lake) e gestire il ripristino dei job e delle tabelle interessate.

L'obiettivo didattico è comprendere e padroneggiare il **ciclo agentico**:

```text
Observe → Decide → Act → Observe → ...
```

```text
            ┌──────────────── Observe (risultato del tool) ◄────────────────┐
            ▼                                                               │
   ┌─────────────────┐        ┌──────────────────┐        ┌────────────────────────┐
   │  Decide         │ ─────► │  Act             │ ─────► │  Environment           │
   │  policy π(a|h)  │        │  tool call       │        │  (job & nodi ETL)      │
   └─────────────────┘        └──────────────────┘        └────────────────────────┘
            │
            └──► Final answer  (condizione di terminazione)
```

### Obiettivi didattici

Al termine del laboratorio sarai in grado di:

1. Distinguere tra **LLM**, **workflow rigido** e **agente autonomo** per flussi Big Data;
2. Progettare **tool** per pipeline ETL come contratti software (input tipizzato → output strutturato);
3. Implementare a mano un **agent loop** con stato, budget di execution turn e condizione di terminazione;
4. Applicare il principio di **minimo privilegio** e il pattern **human-in-the-loop** per azioni correttive (es. restart di job Spark o rilascio lock tabelle);
5. Leggere una **trace** di diagnosi per verificare il *grounding* dell'agente;
6. Riconoscere lo stesso ciclo dentro la **OpenAI Responses API** e l'**Agents SDK**.

### Struttura del notebook

| Livello | Sezioni | API key | Concetto chiave |
|---|---|---|---|
| 1. Ambiente simulato e tools Python | 1 – 6 | No | Tool = contratto; separazione tra monitoraggio (read) e ripristino (write) |
| 2. Agent loop manuale con planner simulato | 7 – 11 | No | Loop, stato, trace, triangolazione evidenze, approvazione umana |
| 3. Integrazione con OpenAI Responses API e Agents SDK | 12 – 13 | Sì (opzionale) | Planner sostituito da un LLM reale con function calling |
| Consolidamento | 14 – 15 | No | Challenge, reset ambiente e debrief |


In [1]:
# Rende "lazy" la valutazione delle annotazioni di tipo (PEP 563)
from __future__ import annotations

import json
import os
from copy import deepcopy
from dataclasses import dataclass, field
from typing import Any, Callable
from dotenv import load_dotenv

print("Ambiente ETL Operations Agent pronto")


Ambiente ETL Operations Agent pronto


## 1. Concetti chiave

Un agente AI per flussi Big Data non coincide con il solo LLM. Utilizziamo la definizione operativa:

$$\text{Agent} = \text{Model} + \text{Tools} + \text{State} + \text{ControlLoop}$$

| Componente | Ruolo | Dove si trova nel notebook |
|---|---|---|
| **Model** | Sceglie la prossima azione di diagnosi o la risposta finale | `simulated_planner` (sez. 7), poi LLM (sez. 12–13) |
| **Tools** | Canale esclusivo per ispezionare job PySpark, log e tabelle o riavviare nodi | Funzioni delle sez. 3–4 |
| **State** | Memoria di lavoro: evento/incident, osservazioni raccolte, diagnosi finale | Dataclass `AgentState` (sez. 7) |
| **ControlLoop** | Orchestratore dei turni, esegue i tool, controlla il budget e lo stop | `run_manual_agent` (sez. 8) |

### Formalizzazione
Il modello seleziona un'azione in base alla storia dell'incidente:

$$a_t \sim \pi(a \mid h_t)$$

Dove:
- $a_t$ è l'azione scelta al turno $t$ (chiamata a un tool di ispezione o risposta finale);
- $h_t$ è la storia delle osservazioni sui job e sulle tabelle;
- $\pi$ rappresenta la policy di diagnosi.


## 2. Ambiente simulato (ETL & Big Data Pipeline)

L'**ambiente** rappresenta il nostro cluster Big Data ed è composto da 3 nodi/step della pipeline:

- `extract-orders`: nodo di Ingestion / Extraction in PySpark (legge da fonti esterne e scrive su tabella di staging);
- `transform-sales`: nodo di Trasformazione in Spark SQL (elabora le vendite partendo dalla tabella di staging);
- `load-dw`: nodo di Caricamento finale in Data Warehouse / Delta Lake.

### Scenario dell'incidente ETL

```text
  Fonti Dati ──► extract-orders (PySpark) ──► transform-sales (Spark SQL) ──► load-dw (Delta Lake)
                        CAUSA                            SINTOMO                    IDLE
                 Lock/Deadlock stg_orders         Attesa/Timeout dipendenza
```

| Nodo | Ruolo | CPU % | RAM % | Servizio | Segnale nei log | Metrica specifica | Diagnosi |
|---|---|---|---|---|---|---|---|
| `extract-orders` | Ingestion | 42 | 58 | `pyspark_job` | `ERROR LockAcquisitionException` | Lock attivo su `stg_orders` | **Causa Radice** |
| `transform-sales` | Processing | 25 | 45 | `spark_sql` | `WARNING Waiting for upstream batch` | Timeout dipendenza dati | **Sintomo** |
| `load-dw` | Loading | 15 | 30 | `delta_loader` | `INFO Idle` | Nessuna anomalia | In attesa |


In [2]:
# STATO INIZIALE DELL'PIPELINE (fixture)
INITIAL_PIPELINES = {
    "extract-orders": {
        "role": "ingestion",
        "status": "failed",
        "cpu": 42,
        "memory": 58,
        "services": {"pyspark_job": "failed"},
        "logs": [
            "08:00 INFO Batch 402 ingestion started from Source API",
            "08:01 ERROR LockAcquisitionException: Timeout acquiring lock on table stg_orders",
            "08:02 ERROR Job terminated with exit code 1",
        ],
        "table_locks": {"stg_orders": {"locked": True, "holding_pid": 14082, "wait_time_sec": 320}},
    },
    "transform-sales": {
        "role": "transformation",
        "status": "waiting",
        "cpu": 25,
        "memory": 45,
        "services": {"spark_sql": "waiting"},
        "logs": [
            "08:01 INFO Waiting for upstream extraction batch 402 on stg_orders",
            "08:03 WARNING Dependency timeout: stg_orders batch 402 not ready after 180s",
            "08:05 WARNING Downstream dashboard metrics stale",
        ],
    },
    "load-dw": {
        "role": "loading",
        "status": "idle",
        "cpu": 15,
        "memory": 30,
        "services": {"delta_loader": "idle"},
        "logs": [
            "08:00 INFO Loader idle: waiting for transformation batch completion",
        ],
    },
}

pipelines = deepcopy(INITIAL_PIPELINES)
pipelines


{'extract-orders': {'role': 'ingestion',
  'status': 'failed',
  'cpu': 42,
  'memory': 58,
  'services': {'pyspark_job': 'failed'},
  'logs': ['08:00 INFO Batch 402 ingestion started from Source API',
   '08:01 ERROR LockAcquisitionException: Timeout acquiring lock on table stg_orders',
   '08:02 ERROR Job terminated with exit code 1'],
  'table_locks': {'stg_orders': {'locked': True,
    'holding_pid': 14082,
    'wait_time_sec': 320}}},
 'transform-sales': {'role': 'transformation',
  'status': 'waiting',
  'cpu': 25,
  'memory': 45,
  'services': {'spark_sql': 'waiting'},
  'logs': ['08:01 INFO Waiting for upstream extraction batch 402 on stg_orders',
   '08:03 WARNING Dependency timeout: stg_orders batch 402 not ready after 180s',
   '08:05 WARNING Downstream dashboard metrics stale']},
 'load-dw': {'role': 'loading',
  'status': 'idle',
  'cpu': 15,
  'memory': 30,
  'services': {'delta_loader': 'idle'},
  'logs': ['08:00 INFO Loader idle: waiting for transformation batch complet

## 3. Tools read-only

I tool di lettura permettono all'agente di esplorare lo stato della pipeline ETL senza causare alterazioni dei dati.

| Tool | Argomenti | Payload restituito | Uso |
|---|---|---|---|
| `get_pipeline_status` | `node` | `role`, `status`, `services` | Verificare lo stato del nodo Spark/ETL |
| `get_node_metrics` | `node` | `cpu`, `memory` | Escludere saturazione di risorse fisiche |
| `read_execution_log` | `node`, `last_lines=5` | ultime righe di log | Rilevare eccezioni, stack trace o warning |
| `check_table_locks` | `node="extract-orders"` | `locked`, `holding_pid`, `wait_time_sec` | Verificare blocchi o deadlock sulle tabelle di staging |


In [3]:
def get_pipeline_status(node: str) -> dict[str, Any]:
    """Returns status and running services for a pipeline node."""
    if node not in pipelines:
        return {"success": False, "error": "NODE_NOT_FOUND", "node": node}
    p = pipelines[node]
    return {
        "success": True,
        "node": node,
        "role": p["role"],
        "status": p["status"],
        "services": p["services"],
    }

def get_node_metrics(node: str) -> dict[str, Any]:
    """Returns CPU and memory utilization for a pipeline worker node."""
    if node not in pipelines:
        return {"success": False, "error": "NODE_NOT_FOUND", "node": node}
    p = pipelines[node]
    return {
        "success": True,
        "node": node,
        "cpu": p["cpu"],
        "memory": p["memory"],
    }

def read_execution_log(node: str, last_lines: int = 5) -> dict[str, Any]:
    """Returns latest log entries for a pipeline execution node."""
    if node not in pipelines:
        return {"success": False, "error": "NODE_NOT_FOUND", "node": node}
    return {
        "success": True,
        "node": node,
        "logs": pipelines[node]["logs"][-last_lines:],
    }

def check_table_locks(node: str = "extract-orders") -> dict[str, Any]:
    """Returns table lock status and holding PIDs for staging tables."""
    if node not in pipelines:
        return {"success": False, "error": "NODE_NOT_FOUND", "node": node}
    if "table_locks" not in pipelines[node]:
        return {"success": False, "error": "NO_TABLE_LOCKS_INFO", "node": node}
    
    locks = pipelines[node]["table_locks"]
    return {
        "success": True,
        "node": node,
        "locks": locks,
    }

# Smoke test dei tool
print(get_pipeline_status("transform-sales"))
print(read_execution_log("extract-orders", last_lines=2))
print(check_table_locks("extract-orders"))


{'success': True, 'node': 'transform-sales', 'role': 'transformation', 'status': 'waiting', 'services': {'spark_sql': 'waiting'}}
{'success': True, 'node': 'extract-orders', 'logs': ['08:01 ERROR LockAcquisitionException: Timeout acquiring lock on table stg_orders', '08:02 ERROR Job terminated with exit code 1']}
{'success': True, 'node': 'extract-orders', 'locks': {'stg_orders': {'locked': True, 'holding_pid': 14082, 'wait_time_sec': 320}}}


## 4. Tool con side effect

I riavvii dei nodi ETL o il riprocessamento delle tabelle comportano **modifiche di stato** (scrittura o rigenerazione dati) e richiedono un gate di approvazione umano (*Human-in-the-loop*).


In [4]:
def restart_pipeline_node(node: str, approved: bool = False) -> dict[str, Any]:
    """Restarts a failed or stuck pipeline node. Requires explicit human approval."""
    if not approved:
        return {
            "success": False,
            "requires_approval": True,
            "message": f"Restarting pipeline node '{node}' requires explicit human approval.",
        }

    if node not in pipelines:
        return {"success": False, "error": "NODE_NOT_FOUND", "node": node}

    pipelines[node]["status"] = "running"
    pipelines[node]["services"]["pyspark_job"] = "running"
    pipelines[node]["logs"].append(f"08:10 INFO Node {node} restarted by AI ETL operator")

    return {
        "success": True,
        "node": node,
        "status": "restarted",
    }

# Prova senza approvazione
restart_pipeline_node("extract-orders", approved=False)


{'success': False,
 'requires_approval': True,
 'message': "Restarting pipeline node 'extract-orders' requires explicit human approval."}

## 5. Tool registry

Tabella di dispatch (*allowlist*) che associa i nomi delle stringhe prodotte dal modello alle funzioni Python reali.


In [5]:
TOOL_REGISTRY: dict[str, Callable[..., dict[str, Any]]] = {
    "get_pipeline_status": get_pipeline_status,
    "get_node_metrics": get_node_metrics,
    "read_execution_log": read_execution_log,
    "check_table_locks": check_table_locks,
    "restart_pipeline_node": restart_pipeline_node,
}

list(TOOL_REGISTRY.keys())


['get_pipeline_status',
 'get_node_metrics',
 'read_execution_log',
 'check_table_locks',
 'restart_pipeline_node']

## 6. Tool schema per un LLM

JSON Schema compatibile con OpenAI function calling in *strict mode*.


In [6]:
tools_schema = [
    {
        "type": "function",
        "name": "get_pipeline_status",
        "description": "Returns status, role and services for a pipeline node.",
        "parameters": {
            "type": "object",
            "properties": {"node": {"type": "string"}},
            "required": ["node"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "get_node_metrics",
        "description": "Returns CPU and RAM utilization for a worker node.",
        "parameters": {
            "type": "object",
            "properties": {"node": {"type": "string"}},
            "required": ["node"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "read_execution_log",
        "description": "Returns recent log lines for a pipeline node. Always specify last_lines (default 5).",
        "parameters": {
            "type": "object",
            "properties": {
                "node": {"type": "string"},
                "last_lines": {"type": "integer"},
            },
            "required": ["node", "last_lines"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "check_table_locks",
        "description": "Returns table lock details and holding PIDs for staging tables.",
        "parameters": {
            "type": "object",
            "properties": {"node": {"type": "string"}},
            "required": ["node"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

tools_schema_safe = tools_schema
print("Tool sicuri esposti all'agente:", [t["name"] for t in tools_schema_safe])


Tool sicuri esposti all'agente: ['get_pipeline_status', 'get_node_metrics', 'read_execution_log', 'check_table_locks']


## 7. Planner simulato deterministico

Implementazione del modello decisionale deterministico $\pi(a \mid h_t)$ per simulare la diagnosi dell'incidente ETL senza dipendere da chiamate API estere.


In [7]:
@dataclass
class ToolCall:
    name: str
    arguments: dict[str, Any]

@dataclass
class AgentState:
    user_request: str
    observations: list[dict[str, Any]] = field(default_factory=list)
    final_answer: str | None = None

def simulated_planner(state: AgentState) -> ToolCall | str:
    called = [obs.get("tool") for obs in state.observations]

    # Step 1: Ispezionare il nodo dove si manifesta il problema segnalato (transform-sales)
    if "get_node_metrics:transform-sales" not in called:
        return ToolCall("get_node_metrics", {"node": "transform-sales"})

    # Step 2: Leggere i log di transform-sales
    if "read_execution_log:transform-sales" not in called:
        return ToolCall("read_execution_log", {"node": "transform-sales", "last_lines": 5})

    # Step 3: Risalire al nodo a monte (extract-orders) e leggere i log
    if "read_execution_log:extract-orders" not in called:
        return ToolCall("read_execution_log", {"node": "extract-orders", "last_lines": 5})

    # Step 4: Verificare la presenza di lock sulle tabelle di staging
    if "check_table_locks:extract-orders" not in called:
        return ToolCall("check_table_locks", {"node": "extract-orders"})

    # Diagnosi finale basata sull'evidenza
    return (
        "Causa radice identificata: Blocco (LockAcquisitionException) sulla tabella di staging 'stg_orders'. "
        "Il job PySpark 'extract-orders' è fallito al tentativo di acquisire il lock (holding PID: 14082, attesa 320s). "
        "Di conseguenza, il nodo a monte 'transform-sales' è in stato di timeout per mancanza di dati freschi. "
        "Azione consigliata: Rilasciare il lock sulla tabella stg_orders e richiedere il riavvio approvato del nodo extract-orders."
    )


## 8. Agent loop manuale

Ciclo di controllo `Observe → Decide → Act` con budget di turni limitato.


In [8]:
def execute_tool_call(call: ToolCall) -> dict[str, Any]:
    if call.name not in TOOL_REGISTRY:
        return {"success": False, "error": "UNKNOWN_TOOL", "tool": call.name}
    func = TOOL_REGISTRY[call.name]
    return func(**call.arguments)

def run_manual_agent(user_request: str, max_turns: int = 8) -> AgentState:
    state = AgentState(user_request=user_request)

    for turn in range(1, max_turns + 1):
        decision = simulated_planner(state)

        if isinstance(decision, str):
            state.final_answer = decision
            print(f"\nFINAL DIAGNOSIS:\n{decision}")
            return state

        print(f"TURN {turn}: calling {decision.name}({decision.arguments})")
        result = execute_tool_call(decision)
        print("RESULT:", result)

        key_node = decision.arguments.get("node", "")
        state.observations.append({
            "tool": f"{decision.name}:{key_node}",
            "arguments": decision.arguments,
            "result": result,
        })

    state.final_answer = "MAX_TURNS raggiunto: diagnosi incompleta."
    return state

state = run_manual_agent("La pipeline ETL delle vendite è bloccata e le dashboard non si aggiornano da 15 minuti.")


TURN 1: calling get_node_metrics({'node': 'transform-sales'})
RESULT: {'success': True, 'node': 'transform-sales', 'cpu': 25, 'memory': 45}
TURN 2: calling read_execution_log({'node': 'transform-sales', 'last_lines': 5})
RESULT: {'success': True, 'node': 'transform-sales', 'logs': ['08:01 INFO Waiting for upstream extraction batch 402 on stg_orders', '08:03 WARNING Dependency timeout: stg_orders batch 402 not ready after 180s', '08:05 WARNING Downstream dashboard metrics stale']}
TURN 3: calling read_execution_log({'node': 'extract-orders', 'last_lines': 5})
RESULT: {'success': True, 'node': 'extract-orders', 'logs': ['08:00 INFO Batch 402 ingestion started from Source API', '08:01 ERROR LockAcquisitionException: Timeout acquiring lock on table stg_orders', '08:02 ERROR Job terminated with exit code 1']}
TURN 4: calling check_table_locks({'node': 'extract-orders'})
RESULT: {'success': True, 'node': 'extract-orders', 'locks': {'stg_orders': {'locked': True, 'holding_pid': 14082, 'wait_t

## 9. Trace di esecuzione

Visualizzazione ordinata dell'intera storia $h_T$.


In [9]:
def print_trace(state: AgentState) -> None:
    print("USER INCIDENT REPORT:")
    print(state.user_request)
    print("\nEXECUTION TRACE:")
    for i, obs in enumerate(state.observations, start=1):
        print(f"{i}. {obs['tool']} -> {obs['result']}")
    print("\nFINAL ANSWER:")
    print(state.final_answer)

print_trace(state)


USER INCIDENT REPORT:
La pipeline ETL delle vendite è bloccata e le dashboard non si aggiornano da 15 minuti.

EXECUTION TRACE:
1. get_node_metrics:transform-sales -> {'success': True, 'node': 'transform-sales', 'cpu': 25, 'memory': 45}
2. read_execution_log:transform-sales -> {'success': True, 'node': 'transform-sales', 'logs': ['08:01 INFO Waiting for upstream extraction batch 402 on stg_orders', '08:03 WARNING Dependency timeout: stg_orders batch 402 not ready after 180s', '08:05 WARNING Downstream dashboard metrics stale']}
3. read_execution_log:extract-orders -> {'success': True, 'node': 'extract-orders', 'logs': ['08:00 INFO Batch 402 ingestion started from Source API', '08:01 ERROR LockAcquisitionException: Timeout acquiring lock on table stg_orders', '08:02 ERROR Job terminated with exit code 1']}
4. check_table_locks:extract-orders -> {'success': True, 'node': 'extract-orders', 'locks': {'stg_orders': {'locked': True, 'holding_pid': 14082, 'wait_time_sec': 320}}}

FINAL ANSWER

## 10. Calcolo dell'evidenza e Triangolazione

Analisi quantitativa dei tempi di attesa del lock sulla tabella `stg_orders` e confronto con la soglia di timeout della pipeline.


In [10]:
locks = check_table_locks("extract-orders")["locks"]["stg_orders"]
wait_time = locks["wait_time_sec"]
holding_pid = locks["holding_pid"]

print(f"Tabella stg_orders bloccata da PID {holding_pid} da {wait_time} secondi (Soglia max: 180s).")


Tabella stg_orders bloccata da PID 14082 da 320 secondi (Soglia max: 180s).


## 11. Human-in-the-loop (HITL)

Riavvio autorizzato del nodo di Ingestion a valle dell'ispezione umana.


In [11]:
# Fase 1: Proposta senza approvazione
proposal = restart_pipeline_node("extract-orders", approved=False)
print("Risposta proposta senza gate:", proposal)

# Fase 2: Autorizzazione umana
human_approval = True

if human_approval:
    execution = restart_pipeline_node("extract-orders", approved=True)
    print("Esito esecuzione autorizzata:", execution)
else:
    print("Operazione cancellata dall'operatore.")


Risposta proposta senza gate: {'success': False, 'requires_approval': True, 'message': "Restarting pipeline node 'extract-orders' requires explicit human approval."}
Esito esecuzione autorizzata: {'success': True, 'node': 'extract-orders', 'status': 'restarted'}


## 12. Opzionale: OpenAI Responses API con function calling

Esecuzione del ciclo agentico integrando un LLM reale (es. GPT-4o / GPT-5) tramite `client.responses.create`.


In [12]:
load_dotenv()

def run_openai_responses_agent(user_request: str, max_turns: int = 8):
    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY non impostata: salto la demo API.")
        return None

    from openai import OpenAI
    client = OpenAI()
    model = os.getenv("OPENAI_MODEL", "gpt-4o")

    instructions = """
    You are an expert Big Data & ETL Operations Diagnostic Agent.
    Investigate pipeline failures, check node logs and table locks.
    Do not perform node restarts without human authorization.
    """

    input_items = [{"role": "user", "content": user_request}]

    for turn in range(max_turns):
        response = client.responses.create(
            model=model,
            instructions=instructions,
            input=input_items,
            tools=tools_schema_safe,
        )

        input_items.extend(item.model_dump(exclude_none=True) for item in response.output)
        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            print("DIAGNOSIS:", response.output_text)
            return response.output_text

        for call in tool_calls:
            args = json.loads(call.arguments)
            result = TOOL_REGISTRY[call.name](**args)
            print(f"TOOL: {call.name}({args}) -> {result}")
            input_items.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result),
            })

    print("MAX_TURNS raggiunto")
    return None

run_openai_responses_agent("La pipeline ETL delle vendite è bloccata e le dashboard non si aggiornano da 15 minuti.")


OPENAI_API_KEY non impostata: salto la demo API.


## 13. Opzionale: OpenAI Agents SDK

Adozione dell'SDK ufficiale per la gestione nativa del ciclo agentico.


In [13]:
async def run_agents_sdk_demo():
    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY non impostata: salto la demo Agents SDK.")
        return None

    try:
        from agents import Agent, Runner, function_tool
    except Exception as exc:
        print("Installa openai-agents: pip install openai-agents")
        return None

    @function_tool
    def node_metrics(node: str) -> dict:
        """Return CPU and memory utilization for an ETL node."""
        return get_node_metrics(node)

    @function_tool
    def execution_logs(node: str, last_lines: int = 5) -> dict:
        """Return latest log entries for an ETL node."""
        return read_execution_log(node, last_lines)

    @function_tool
    def table_locks(node: str = "extract-orders") -> dict:
        """Return table lock details for staging tables."""
        return check_table_locks(node)

    agent = Agent(
        name="ETL Operations Agent",
        instructions="Diagnose Big Data pipeline failures using tools before concluding.",
        tools=[node_metrics, execution_logs, table_locks],
    )

    result = await Runner.run(
        agent,
        "La pipeline ETL delle vendite è bloccata. Diagnostica la causa.",
        max_turns=8,
    )

    print(result.final_output)
    return result

# await run_agents_sdk_demo()


## 14. Challenge e Reset dell'ambiente

Reset dell'ambiente di prova e verifica della diagnosi su un nuovo report di incidente.


In [14]:
def reset_environment():
    global pipelines
    pipelines = deepcopy(INITIAL_PIPELINES)
    return pipelines

reset_environment()
challenge_state = run_manual_agent("Segnalato ritardo critico sul caricamento delle vendite giornaliere.")
print_trace(challenge_state)


TURN 1: calling get_node_metrics({'node': 'transform-sales'})
RESULT: {'success': True, 'node': 'transform-sales', 'cpu': 25, 'memory': 45}
TURN 2: calling read_execution_log({'node': 'transform-sales', 'last_lines': 5})
RESULT: {'success': True, 'node': 'transform-sales', 'logs': ['08:01 INFO Waiting for upstream extraction batch 402 on stg_orders', '08:03 WARNING Dependency timeout: stg_orders batch 402 not ready after 180s', '08:05 WARNING Downstream dashboard metrics stale']}
TURN 3: calling read_execution_log({'node': 'extract-orders', 'last_lines': 5})
RESULT: {'success': True, 'node': 'extract-orders', 'logs': ['08:00 INFO Batch 402 ingestion started from Source API', '08:01 ERROR LockAcquisitionException: Timeout acquiring lock on table stg_orders', '08:02 ERROR Job terminated with exit code 1']}
TURN 4: calling check_table_locks({'node': 'extract-orders'})
RESULT: {'success': True, 'node': 'extract-orders', 'locks': {'stg_orders': {'locked': True, 'holding_pid': 14082, 'wait_t

## 15. Debrief & Riepilogo dei concetti

### Domande di riflessione per l'aula:
1. Quali vantaggi offre l'isolamento del tool di scrittura (`restart_pipeline_node`) tramite Human-in-the-loop nei contesti Big Data?
2. In che modo l'analisi dei lock sulle tabelle ha permesso di distinguere il *sintomo* dal *problema radice*?
3. Perché l'uso di risposte strutturate e tipizzate nei tool previene fallimenti nel ciclo dell'agente?

| Concetto | Definizione sintetica |
|---|---|
| **Agent Loop** | Ciclo `Observe → Decide → Act` con stato e budget |
| **Tool Contract** | Interfaccia con tipi definiti e gestione errori come dati |
| **Human-in-the-Loop** | Separazione tra la proposta dell'agente e l'autorizzazione di scrittura |
| **Grounding** | Verifica che la diagnosi sia strettamente supportata dalle osservazioni nella trace |
